In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [8]:
phrases = [
    "Le match de football était incroyable",
    "Ce film de science-fiction est incroyable",
    "Le joueur de football a marqué un but incroyable pendant le match",
]
for i, p in enumerate(phrases):
    print(f"Phrase {i}: {p}")

Phrase 0: Le match de football était incroyable
Phrase 1: Ce film de science-fiction est incroyable
Phrase 2: Le joueur de football a marqué un but incroyable pendant le match


In [9]:
cv = CountVectorizer()
X_bow = cv.fit_transform(phrases)

mots = cv.get_feature_names_out()
print("Liste des mots extraits (vocabulaire) :")
print(list(mots))
print("\nNombre de mots :", len(mots))
print("Forme de la matrice :", X_bow.shape, "(3 phrases x", len(mots), "mots)")

Liste des mots extraits (vocabulaire) :
['but', 'ce', 'de', 'est', 'fiction', 'film', 'football', 'incroyable', 'joueur', 'le', 'marqué', 'match', 'pendant', 'science', 'un', 'était']

Nombre de mots : 16
Forme de la matrice : (3, 16) (3 phrases x 16 mots)


In [10]:
print("Matrice Bag of Words (tableau numpy) :")
print(X_bow.toarray())

df_bow = pd.DataFrame(X_bow.toarray(), columns=mots, index=["Phrase 0", "Phrase 1", "Phrase 2"])
df_bow

Matrice Bag of Words (tableau numpy) :
[[0 0 1 0 0 0 1 1 0 1 0 1 0 0 0 1]
 [0 1 1 1 1 1 0 1 0 0 0 0 0 1 0 0]
 [1 0 1 0 0 0 1 1 1 2 1 1 1 0 1 0]]


,but,ce,de,est,fiction,film,football,incroyable,joueur,le,marqué,match,pendant,science,un,était
Phrase 0,0,0,1,0,0,0,1,1,0,1,0,1,0,0,0,1
Phrase 1,0,1,1,1,1,1,0,1,0,0,0,0,0,1,0,0
Phrase 2,1,0,1,0,0,0,1,1,1,2,1,1,1,0,1,0


### Questions — Partie 5

**1. Que représente chaque colonne de la matrice ?**

Chaque colonne correspond à **un mot unique du vocabulaire** (tous les mots distincts trouvés dans l'ensemble des phrases, classés par ordre alphabétique). Chaque ligne correspond à une phrase, et chaque case contient le **nombre de fois** où ce mot apparaît dans la phrase. Par exemple, la colonne « match » vaut 2 pour la phrase 0, car ce mot y apparaît deux fois.

**2. Quelles sont les limites du modèle Bag of Words ?**

- Il **ignore l'ordre des mots** et la structure de la phrase.
- Il **ne comprend pas le sens** : « bon » et « excellent » sont deux colonnes sans lien entre elles (pas de synonymes ni de contexte).
- Les mots très fréquents (« le », « de », « et ») **dominent** la matrice alors qu'ils apportent peu d'information.
- La matrice est **grande et creuse** (beaucoup de zéros) quand le vocabulaire grandit.
- Un mot absent du vocabulaire d'entraînement est **ignoré** lors de la prédiction.

**3. Que perd-on en supprimant l'ordre des mots ?**

On perd le **sens construit par l'ordre** : « le chien mord l'homme » et « l'homme mord le chien » donnent exactement le même vecteur. On perd aussi les **négations** (« pas bon » devient « pas » + « bon »), les expressions (« pomme de terre ») et les relations entre les mots (qui fait quoi).

## Partie 6 — Représentation TF-IDF
On reprend la même liste de phrases, avec `TfidfVectorizer` à la place de `CountVectorizer`.

In [11]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(phrases)

print("Mots extraits :", list(tfidf.get_feature_names_out()))
print("Forme de la matrice :", X_tfidf.shape)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(3),
    columns=tfidf.get_feature_names_out(),
    index=["Phrase 0", "Phrase 1", "Phrase 2"],
)
df_tfidf

Mots extraits : ['but', 'ce', 'de', 'est', 'fiction', 'film', 'football', 'incroyable', 'joueur', 'le', 'marqué', 'match', 'pendant', 'science', 'un', 'était']
Forme de la matrice : (3, 16)


,but,ce,de,est,fiction,film,football,incroyable,joueur,le,marqué,match,pendant,science,un,était
Phrase 0,0.00,0.000,0.319,0.000,0.000,0.000,0.410,0.319,0.00,0.410,0.00,0.410,0.00,0.000,0.00,0.54
Phrase 1,0.00,0.419,0.247,0.419,0.419,0.419,0.000,0.247,0.00,0.000,0.00,0.000,0.00,0.419,0.00,0.00
Phrase 2,0.33,0.000,0.195,0.000,0.000,0.000,0.251,0.195,0.33,0.502,0.33,0.251,0.33,0.000,0.33,0.00


### Comparaison des poids : BoW vs TF-IDF

In [12]:
idf = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out(), name="IDF").round(3)
nb_phrases = (X_bow.toarray() > 0).sum(axis=0)
tableau = pd.DataFrame({"Nb de phrases contenant le mot": nb_phrases}, index=mots).join(idf)
tableau.sort_values("IDF")

,Nb de phrases contenant le mot,IDF
de,3,1.000
incroyable,3,1.000
football,2,1.288
le,2,1.288
match,2,1.288
but,1,1.693
ce,1,1.693
est,1,1.693
fiction,1,1.693
film,1,1.693


In [7]:
comparaison = pd.DataFrame({
    "BoW (comptage)": df_bow.loc["Phrase 0"],
    "TF-IDF (poids)": df_tfidf.loc["Phrase 0"],
})
comparaison = comparaison[comparaison["BoW (comptage)"] > 0].sort_values("TF-IDF (poids)", ascending=False)
comparaison

,BoW (comptage),TF-IDF (poids)
le,3,0.649
match,2,0.432
adoré,1,0.284
public,1,0.284
était,1,0.284
et,1,0.216
football,1,0.216
de,1,0.168
incroyable,1,0.168


### Questions — Partie 6

**1. Quelle est la différence principale entre BoW et TF-IDF ?**

Le BoW **compte simplement** les occurrences d'un mot dans une phrase. TF-IDF **pondère** ce comptage : il multiplie la fréquence du mot dans le document (TF) par un facteur qui mesure sa **rareté dans l'ensemble du corpus** (IDF). Un mot qui apparaît dans tous les documents est pénalisé, un mot rare et caractéristique d'un document est valorisé. En plus, `TfidfVectorizer` normalise chaque ligne, donc les vecteurs ont tous la même longueur.

**2. Pourquoi les mots très fréquents ont-ils souvent un poids faible ?**

Parce qu'un mot présent dans presque tous les documents (« de », « et », ici « incroyable ») ne permet **pas de distinguer** un document d'un autre. Son IDF est donc proche de son minimum, ce qui réduit son poids final. Dans notre exemple, « de » et « incroyable » sont dans les 3 phrases : leur IDF est le plus bas (1.0), alors que « acteurs » ou « science » (présents dans une seule phrase) ont un IDF plus élevé (1.69).

**Remarque :** avec seulement 3 phrases, « le » garde un poids élevé dans la phrase 0 (0.649), car il y apparaît 3 fois : le TF compense l'IDF. Sur un grand corpus, ou en supprimant les stopwords, ce problème disparaît.

**3. Exemple concret où TF-IDF serait plus utile que BoW**

Dans un **moteur de recherche** ou un classement d'articles : si on cherche « football » parmi des milliers d'articles, un mot comme « le » apparaît partout et fausserait la pertinence avec le BoW. Avec TF-IDF, les articles contenant souvent « football » (mot rare dans le corpus) remontent en premier. Même idée pour la **classification de textes** (sport vs cinéma) ou la détection de **spam** : les mots distinctifs pèsent plus que les mots courants.